In [2]:
import os
import sys
from pathlib import Path

os.environ['SONG_ENTRY_DB_PATH'] = str(Path('..') / 'data' / 'database' / 'song_bureaucracy_entries_v0304_run2.db')

AGENT_DIR = Path.cwd()
if str(AGENT_DIR) not in sys.path:
    sys.path.insert(0, str(AGENT_DIR))


In [3]:
from dotenv import load_dotenv
load_dotenv()
from llm_client import SimpleLLMClient, LLMTool
from database import Database
from agent_state import AgentState
from config import DICT_DB_PATH, DICT_TABLE, ENTRY_DB_PATH, ensure_save_dir, validate_paths

model_name = os.getenv('OPENROUTER_MODEL', 'deepseek/deepseek-v4-flash')
max_tokens = int(os.getenv('OPENROUTER_MAX_TOKENS', '16384'))
llm = SimpleLLMClient(model=model_name, max_tokens=max_tokens)
llm_tool = LLMTool(model_name, llm)
llm_tool.compile()

validate_paths()
SAVE_DIR = ensure_save_dir()
db = Database(str(DICT_DB_PATH), DICT_TABLE, str(ENTRY_DB_PATH))
print(f'LLM: {model_name}')
print(f'DB: {ENTRY_DB_PATH}')


LLM: qwen/qwen3-max
DB: ../data/database/song_bureaucracy_entries_v0304_run2.db


In [ ]:
import copy
import time
import json

dict_index_list = db.get_dictionary_index()
dict_index_text = "\n".join(dict_index_list)
state = AgentState(db=db, dict_index_text=dict_index_text)
todo_dict_entries = dict_index_list[:50]
print(f"Total entries: {len(dict_index_list)}, Processing: {len(todo_dict_entries)}")

tools_input2facts = {
  "search_dictionary": state.tool_search_dictionary,
  "add_atomic_fact": state.tool_add_atomic_fact,
  "remove_atomic_fact": state.tool_remove_atomic_fact,
  "update_atomic_fact": state.tool_update_atomic_fact,
}

tools_facts2data = {
  "get_entity": state.tool_get_entity,
  "create_entity": state.tool_create_entity,
  "create_timepoint": state.tool_create_timepoint,
  "update_timepoint_attr": state.tool_update_timepoint_attr,
  "create_timepoints_relationship": state.tool_create_timepoints_relationship,
  "append_citation": state.tool_append_citation
}

MAX_llm_loop_count = 10
failed_entries = []
overall_start = time.time()

for i, entry_index in enumerate(todo_dict_entries):
    entry_start = time.time()
    print(f"\n{'='*50}")
    print(f"[{i+1}/{len(todo_dict_entries)}] {entry_index}")
    
    try:
        records = {"input2facts": {}, "facts2data": {}}
        
        state.prepare_new_round()
        state.append_input_entry(entry_index)
        records["input2facts"]["prompts"] = []
        
        loop_count = 0
        while not state.finished_facts and loop_count < MAX_llm_loop_count:
            loop_count += 1
            prompt = state.build_prompt_input2facts()
            records["input2facts"]["prompts"].append(prompt)
            response = llm_tool.invoke({"prompt": prompt})
            content = response["response"]["response"]["choices"][0]["message"]["content"]
            flag = state.parse_cot(content, tools_input2facts)
            state.finished_facts = flag
        
        facts_count = len(state.atomic_facts.facts)
        print(f"  Stage1: {loop_count} loops, {facts_count} facts")
        records["input2facts"]["cot"] = copy.deepcopy(state.cot.chain)
        records["input2facts"]["atomic_facts"] = copy.deepcopy(state.atomic_facts.get_merged_text())
        
        state.prepare_for_update()
        records["facts2data"]["prompts"] = []
        
        loop_count = 0
        while not state.finished_update and loop_count < MAX_llm_loop_count:
            loop_count += 1
            prompt = state.build_prompt_facts2data()
            records["facts2data"]["prompts"].append(prompt)
            response = llm_tool.invoke({"prompt": prompt})
            content = response["response"]["response"]["choices"][0]["message"]["content"]
            flag = state.parse_cot(content, tools_facts2data)
            state.finished_update = flag
        
        items_count = len(state.loaded_data_items.items)
        print(f"  Stage2: {loop_count} loops, {items_count} items")
        records["facts2data"]["cot"] = copy.deepcopy(state.cot.chain)
        records["facts2data"]["loaded_data_items"] = copy.deepcopy(state.loaded_data_items.get_merged_text())
        
        save_path = SAVE_DIR / f"records_{i+1}_{entry_index}.json"
        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(records, f, indent=2, ensure_ascii=False)
        
        elapsed = time.time() - entry_start
        print(f"  Saved: {save_path.name} ({elapsed:.1f}s)")
        
    except Exception as e:
        print(f"  [FAILED] {entry_index}: {e}")
        failed_entries.append(entry_index)

overall = time.time() - overall_start
print(f"\n{'='*50}")
print(f"Done: {len(todo_dict_entries)-len(failed_entries)}/{len(todo_dict_entries)} in {overall:.1f}s")
print(f"Failed: {len(failed_entries)}")


Total entries: 833, Processing: 50

[1/50] 河北兵马大元帅府-482
  Stage1: 2 loops, 7 facts
